
# 28 Final Scenario Review (Hourly + Quarter-Hour)

This notebook is the final joint review for scenario generation across:

- hourly DA scenarios (legacy hourly scenario notebook artifacts)
- quarter-hour DA scenarios (new three-model observed-QH finalisation run)

It answers three questions:
1. Do we have scenarios for all models or only selected candidates?
2. What are the scenario coverage and point-forecast accuracy on a selected week?
3. Are outputs ready for MILP ingestion?

All line charts use a light background; the actual line is rendered in high-contrast red.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive, and thus cannot be shown')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['grid.color'] = '#cccccc'


## Locate Latest Runs

## Load Scenario Artifacts

In [ ]:
def resolve_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() or (candidate / '.git').exists():
            return candidate
    return Path.cwd().resolve()

PROJECT_ROOT = resolve_project_root()

# Prefer the LEAR_STRICT-extended hourly scenario run root; fallback to legacy hourly scenario root.
hourly_root_candidates = [
    PROJECT_ROOT / 'data/02_Forecasting/01_DA_prices/hourly_da/notebook_artifacts/01_da_price_scenario_generation_hourly_with_lear_strict',
    PROJECT_ROOT / 'data/02_Forecasting/01_DA_prices/hourly_da/notebook_artifacts/01_da_price_scenario_generation_hourly',
]
hourly_root = next((root for root in hourly_root_candidates if root.exists()), hourly_root_candidates[-1])

# QH scenario artifacts root
qh_root = PROJECT_ROOT / 'data/02_Forecasting/01_DA_prices/quarterhour_da/finalisation_runs/qh_scenario_generation'

def run_has_files(run_dir, required_files, optional_any=None):
    if not run_dir.is_dir():
        return False
    for fname in required_files:
        if not (run_dir / fname).exists():
            return False
    if optional_any:
        for alternatives in optional_any:
            if not any((run_dir / alt).exists() for alt in alternatives):
                return False
    return True

def run_status(run_dir):
    summary_path = run_dir / 'scenario_generation_run_summary.json'
    if not summary_path.exists():
        return None
    try:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        return str(summary.get('status', '')).lower()
    except Exception:
        return None

def latest_complete_run(root, required_files, optional_any=None):
    runs = [p for p in root.iterdir() if p.is_dir()]
    complete = [p for p in runs if run_has_files(p, required_files, optional_any=optional_any)]
    if not complete:
        raise FileNotFoundError(f'No complete run found under {root} for required files: {required_files}')

    preferred = []
    for run_dir in complete:
        status = run_status(run_dir)
        name = run_dir.name.lower()
        if status == 'completed' and 'smoke' not in name and 'check' not in name:
            preferred.append(run_dir)

    pool = preferred if preferred else complete
    return sorted(pool)[-1]

hourly_run = latest_complete_run(
    hourly_root,
    required_files=[
        'scenario_generation_run_summary.json',
        'scenario_generation_config.json',
        'scenario_prices_long.csv',
        'scenario_period_quantiles.csv',
        'scenario_validation_summary.csv',
    ],
)
qh_run = latest_complete_run(
    qh_root,
    required_files=[
        'scenario_generation_run_summary.json',
        'scenario_generation_config.json',
        'scenario_period_quantiles.csv',
        'scenario_validation_summary.csv',
    ],
    optional_any=[['scenario_prices_long.parquet', 'scenario_prices_long.csv']],
)

hourly_run, qh_run


In [ ]:
# Hourly
hourly_summary = json.loads((hourly_run / 'scenario_generation_run_summary.json').read_text(encoding='utf-8'))
hourly_cfg = json.loads((hourly_run / 'scenario_generation_config.json').read_text(encoding='utf-8'))
hourly_quantiles = pd.read_csv(hourly_run / 'scenario_period_quantiles.csv')
hourly_validation = pd.read_csv(hourly_run / 'scenario_validation_summary.csv')
hourly_reco_path = hourly_run / 'final_scenario_recommendation.json'
hourly_reco = json.loads(hourly_reco_path.read_text(encoding='utf-8')) if hourly_reco_path.exists() else {}

# QH
qh_summary = json.loads((qh_run / 'scenario_generation_run_summary.json').read_text(encoding='utf-8'))
qh_cfg = json.loads((qh_run / 'scenario_generation_config.json').read_text(encoding='utf-8'))
qh_quantiles = pd.read_csv(qh_run / 'scenario_period_quantiles.csv')
qh_validation = pd.read_csv(qh_run / 'scenario_validation_summary.csv')
qh_support_by_model_path = qh_run / 'support_by_model.csv'
qh_support_by_model = pd.read_csv(qh_support_by_model_path) if qh_support_by_model_path.exists() else pd.DataFrame()


## Availability: Do We Have Scenarios For All Models?

In [ ]:
hourly_candidates = sorted(
    pd.Series([record.get('candidate_key') for record in hourly_summary.get('selected_candidates', [])])
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)
qh_models = sorted(
    pd.Series(qh_summary.get('models', []))
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

availability = pd.DataFrame([
    {
        'track': 'hourly',
        'models_with_scenarios': len(hourly_candidates),
        'model_ids_or_keys': ' | '.join(hourly_candidates),
        'note': 'Hourly scenarios are generated for selected candidates from hourly scenario selection, not full model inventory.'
    },
    {
        'track': 'quarter-hour',
        'models_with_scenarios': len(qh_models),
        'model_ids_or_keys': ' | '.join(qh_models),
        'note': 'QH scenarios generated for all three selected observed-QH models.'
    }
])
availability

In [ ]:

# Explicit answer cell
print('Hourly scenario candidate keys:', hourly_candidates)
print('QH scenario model_ids:', qh_models)


## Standardize Time Columns

In [ ]:
# Hourly schema
hourly_quantiles['period_timestamp'] = pd.to_datetime(hourly_quantiles['period_timestamp'], utc=True, errors='coerce')
hourly_quantiles['delivery_day'] = pd.to_datetime(hourly_quantiles['delivery_day'], errors='coerce').dt.date

# QH schema
qh_quantiles['target_timestamp_utc'] = pd.to_datetime(qh_quantiles['target_timestamp_utc'], utc=True, errors='coerce')
if 'target_local_date' in qh_quantiles.columns:
    qh_quantiles['target_local_date'] = pd.to_datetime(qh_quantiles['target_local_date'], errors='coerce').dt.date
else:
    qh_quantiles['target_local_date'] = qh_quantiles['target_timestamp_utc'].dt.tz_convert('Europe/Amsterdam').dt.date


## Select Week(s)

In [ ]:
# Because hourly and QH scenario tracks are on different calendar windows,
# we select one week per track.

HOURLY_SPLIT = 'test'
QH_SPLIT = 'test'

# Fast deterministic week selection: last complete week within each split.
hourly_dates = pd.to_datetime(
    hourly_quantiles.loc[hourly_quantiles['dataset_split'].astype(str).eq(HOURLY_SPLIT), 'delivery_day'],
    errors='coerce'
).dropna()
qh_dates = pd.to_datetime(
    qh_quantiles.loc[qh_quantiles['dataset_split'].astype(str).eq(QH_SPLIT), 'target_local_date'],
    errors='coerce'
).dropna()

if hourly_dates.empty or qh_dates.empty:
    raise ValueError('Selected split has no dates for hourly or QH scenario quantiles.')

hourly_last_day = hourly_dates.max().normalize()
qh_last_day = qh_dates.max().normalize()

hourly_week_start = hourly_last_day - pd.to_timedelta(hourly_last_day.weekday(), unit='D')
qh_week_start = qh_last_day - pd.to_timedelta(qh_last_day.weekday(), unit='D')

print('Selected hourly week start:', hourly_week_start.date(), 'end:', (hourly_week_start + pd.Timedelta(days=6)).date())
print('Selected QH week start:', qh_week_start.date(), 'end:', (qh_week_start + pd.Timedelta(days=6)).date())

## Build Week Subsets

In [ ]:
def within_week(date_series: pd.Series, week_start: pd.Timestamp) -> pd.Series:
    return (pd.to_datetime(date_series) >= week_start) & (pd.to_datetime(date_series) < week_start + pd.Timedelta(days=7))

hourly_week_quantiles = hourly_quantiles[
    hourly_quantiles['dataset_split'].astype(str).eq(HOURLY_SPLIT)
    & within_week(hourly_quantiles['delivery_day'], hourly_week_start)
].copy()

qh_week_quantiles = qh_quantiles[
    qh_quantiles['dataset_split'].astype(str).eq(QH_SPLIT)
    & within_week(qh_quantiles['target_local_date'], qh_week_start)
].copy()

# Reuse naming in downstream cells
hourly_week_prices = hourly_week_quantiles.copy()
qh_week_prices = qh_week_quantiles.copy()

hourly_week_quantiles.shape, qh_week_quantiles.shape

## Weekly Point-Forecast Metrics

In [ ]:

def point_metrics(df: pd.DataFrame, model_col: str, actual_col: str, pred_col: str) -> pd.DataFrame:
    rows=[]
    for model, g in df.groupby(model_col, dropna=False):
        y_true = pd.to_numeric(g[actual_col], errors='coerce')
        y_pred = pd.to_numeric(g[pred_col], errors='coerce')
        mask = y_true.notna() & y_pred.notna()
        if mask.sum() == 0:
            rows.append({model_col:model, 'n':0, 'mae':np.nan, 'rmse':np.nan, 'bias':np.nan, 'median_ae':np.nan, 'p90_ae':np.nan, 'p95_ae':np.nan})
            continue
        err = y_pred[mask]-y_true[mask]
        ae = err.abs()
        rows.append({
            model_col:model,
            'n':int(mask.sum()),
            'mae':float(ae.mean()),
            'rmse':float(np.sqrt((err**2).mean())),
            'bias':float(err.mean()),
            'median_ae':float(ae.median()),
            'p90_ae':float(ae.quantile(0.90)),
            'p95_ae':float(ae.quantile(0.95)),
        })
    return pd.DataFrame(rows).sort_values(model_col).reset_index(drop=True)

hourly_week_point = point_metrics(hourly_week_prices, 'candidate_key', 'actual_price', 'central_forecast_price')
qh_week_point = point_metrics(qh_week_prices, 'model_id', 'y_true', 'point_forecast')

hourly_week_point, qh_week_point


## Weekly Scenario Coverage Metrics

In [ ]:

def scenario_coverage_from_quantiles(df: pd.DataFrame, model_col: str, actual_col: str) -> pd.DataFrame:
    rows=[]
    for model, g in df.groupby(model_col, dropna=False):
        y = pd.to_numeric(g[actual_col], errors='coerce')
        p10 = pd.to_numeric(g['p10'], errors='coerce')
        p90 = pd.to_numeric(g['p90'], errors='coerce')
        p05 = pd.to_numeric(g['p05'], errors='coerce')
        p95 = pd.to_numeric(g['p95'], errors='coerce')
        mask = y.notna() & p10.notna() & p90.notna() & p05.notna() & p95.notna()
        if mask.sum()==0:
            rows.append({model_col:model, 'n':0, 'coverage_p10_p90':np.nan, 'coverage_p05_p95':np.nan, 'avg_width_p10_p90':np.nan, 'avg_width_p05_p95':np.nan})
            continue
        yy = y[mask]
        rows.append({
            model_col:model,
            'n':int(mask.sum()),
            'coverage_p10_p90':float(((p10[mask] <= yy) & (yy <= p90[mask])).mean()),
            'coverage_p05_p95':float(((p05[mask] <= yy) & (yy <= p95[mask])).mean()),
            'avg_width_p10_p90':float((p90[mask]-p10[mask]).mean()),
            'avg_width_p05_p95':float((p95[mask]-p05[mask]).mean()),
        })
    return pd.DataFrame(rows).sort_values(model_col).reset_index(drop=True)

hourly_week_cov = scenario_coverage_from_quantiles(hourly_week_quantiles, 'candidate_key', 'actual_price')
qh_week_cov = scenario_coverage_from_quantiles(qh_week_quantiles, 'model_id', 'y_true')

hourly_week_cov, qh_week_cov


## Visual 1: Weekly Point Accuracy Comparison

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

if not hourly_week_point.empty:
    x = np.arange(len(hourly_week_point))
    axes[0].bar(x, hourly_week_point['mae'], color='#4C78A8')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(hourly_week_point['candidate_key'], rotation=30, ha='right')
    axes[0].set_title('Hourly Weekly MAE')
    axes[0].grid(alpha=0.2, axis='y')

if not qh_week_point.empty:
    x = np.arange(len(qh_week_point))
    axes[1].bar(x, qh_week_point['mae'], color='#F58518')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(qh_week_point['model_id'], rotation=30, ha='right')
    axes[1].set_title('QH Weekly MAE')
    axes[1].grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.show()


## Visual 2: Hourly Week Scenario Bands

In [ ]:
# Use recommended hourly scenario variant when available
recommended_variant = hourly_reco.get('default_scenario_variant', None)
plot_hourly = hourly_week_quantiles.copy()
if recommended_variant is not None and 'scenario_variant' in plot_hourly.columns:
    subset = plot_hourly[plot_hourly['scenario_variant'].astype(str).eq(str(recommended_variant))].copy()
    if not subset.empty:
        plot_hourly = subset

models = sorted(plot_hourly['candidate_key'].dropna().unique().tolist())
fig, axes = plt.subplots(len(models), 1, figsize=(14, 3.2*max(len(models),1)), sharex=True)
if len(models)==1:
    axes = [axes]

for ax, model in zip(axes, models):
    g = plot_hourly[plot_hourly['candidate_key'].astype(str).eq(str(model))].sort_values('period_timestamp').copy()
    for col in ['actual_price', 'p50', 'p10', 'p90', 'p05', 'p95']:
        g[col] = pd.to_numeric(g[col], errors='coerce')
    g = g.dropna(subset=['period_timestamp', 'actual_price', 'p50', 'p10', 'p90', 'p05', 'p95'])
    if g.empty:
        ax.set_title(f'Hourly: {model} (no plottable rows)')
        ax.grid(alpha=0.2)
        continue
    ax.plot(g['period_timestamp'], g['actual_price'], color='#d62728', linewidth=1.8, label='Actual')
    ax.plot(g['period_timestamp'], g['p50'], color='#1f77b4', linewidth=1.2, label='Scenario p50')
    ax.fill_between(g['period_timestamp'], g['p10'], g['p90'], color='#1f77b4', alpha=0.22, label='p10-p90')
    ax.fill_between(g['period_timestamp'], g['p05'], g['p95'], color='#1f77b4', alpha=0.12, label='p05-p95')
    ax.set_title(f'Hourly: {model}')
    ax.grid(alpha=0.2)

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=4)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()


## Visual 3: Quarter-Hour Week Scenario Bands

In [ ]:
plot_qh = qh_week_quantiles.copy()
models = sorted(plot_qh['model_id'].dropna().unique().tolist())
fig, axes = plt.subplots(len(models), 1, figsize=(14, 3.2*max(len(models),1)), sharex=True)
if len(models)==1:
    axes = [axes]

for ax, model in zip(axes, models):
    g = plot_qh[plot_qh['model_id'].astype(str).eq(str(model))].sort_values('target_timestamp_utc').copy()
    for col in ['y_true', 'p50', 'p10', 'p90', 'p05', 'p95']:
        g[col] = pd.to_numeric(g[col], errors='coerce')
    g = g.dropna(subset=['target_timestamp_utc', 'y_true', 'p50', 'p10', 'p90', 'p05', 'p95'])
    if g.empty:
        ax.set_title(f'QH: {model} (no plottable rows)')
        ax.grid(alpha=0.2)
        continue
    ax.plot(g['target_timestamp_utc'], g['y_true'], color='#d62728', linewidth=1.8, label='Actual')
    ax.plot(g['target_timestamp_utc'], g['p50'], color='#2ca02c', linewidth=1.1, label='Scenario p50')
    ax.fill_between(g['target_timestamp_utc'], g['p10'], g['p90'], color='#2ca02c', alpha=0.22, label='p10-p90')
    ax.fill_between(g['target_timestamp_utc'], g['p05'], g['p95'], color='#2ca02c', alpha=0.12, label='p05-p95')
    ax.set_title(f'QH: {model}')
    ax.grid(alpha=0.2)

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=4)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()


## Visual 4: Weekly Scenario Coverage

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

if not hourly_week_cov.empty:
    x = np.arange(len(hourly_week_cov))
    axes[0].bar(x - 0.15, hourly_week_cov['coverage_p10_p90'], width=0.3, label='p10-p90', color='#4C78A8')
    axes[0].bar(x + 0.15, hourly_week_cov['coverage_p05_p95'], width=0.3, label='p05-p95', color='#72B7B2')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(hourly_week_cov['candidate_key'], rotation=30, ha='right')
    axes[0].set_title('Hourly Coverage')
    axes[0].set_ylim(0,1)
    axes[0].grid(alpha=0.2, axis='y')
    axes[0].legend()

if not qh_week_cov.empty:
    x = np.arange(len(qh_week_cov))
    axes[1].bar(x - 0.15, qh_week_cov['coverage_p10_p90'], width=0.3, label='p10-p90', color='#F58518')
    axes[1].bar(x + 0.15, qh_week_cov['coverage_p05_p95'], width=0.3, label='p05-p95', color='#E45756')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(qh_week_cov['model_id'], rotation=30, ha='right')
    axes[1].set_title('QH Coverage')
    axes[1].set_ylim(0,1)
    axes[1].grid(alpha=0.2, axis='y')
    axes[1].legend()

plt.tight_layout()
plt.show()


## Readiness + Key Notes

In [ ]:

readiness = {
    'hourly_run': str(hourly_run),
    'qh_run': str(qh_run),
    'hourly_models_with_scenarios': hourly_candidates,
    'qh_models_with_scenarios': qh_models,
    'qh_milp_compatibility': qh_summary.get('milp_ingestion_compatibility', {}),
    'hourly_selected_candidates_note': 'Hourly scenarios are currently generated for selected candidates only (not all hourly models).'
}
readiness



### Interpretation Guardrails

- Hourly and QH scenario tracks are generated on different calendar windows, so the selected week is chosen independently per track.
- Weekly metrics here are diagnostic and comparison-oriented; final thesis claims should remain on the canonical split/scope summaries.
- For downstream MILP, prefer parquet where available (`scenario_prices_long.parquet`) due file size.
